# Main file 

05.11.2025

Written by: Arnav Kumar Upadhyay, Cedric Meyer adapted from MATLAB code of Jan Spindler DOI-https://doi.org/10.1021/acssynbio.4c00578

This script is the main script for the TXTL v2 model.


In [ ]:
# Packages functions and data types --------
include("packages.jl") 

# Sequences 
sequence_file	    = "Ecoli_K12_MG1655_geneset_with_welch_2009.fa"    # "lacS_background.fa", "Ecoli_K12_MG1655_geneset_cds_hpaBC_ddc.fa", "Ecoli_example_sequences.fa"
hybrid_file		    = "lacS.txt"
seqs_to_use 	    = (1,2323)                                 # Range of sequences in Fasta file. Takes all sequences from sequence a to sequence b
num_background_genes = 0
heterologous_genes  = ["hpaBC","ddc"]

# initialization of data containers
scale = txtl_types.init_scale()         # struct that makes it possible to influence rates by scaling them

µ = 1                    # A growth rate (0.1 to 1.8) can be chosen and the proteins of the TXTL machinery will be tuned according to that growth rate and the processivity of the RNA polymerase

## flags for efficiency factors
positive_aminoacid	= true 
anti_shinedalgarno	= false      # If no hybridization file is available put flag to false. We apply the intaRNA program's intarna function (q: aSD, t: sequence, SeedBP: 5, out: tMinE) to generate a hybridization profile along the sequence. Link to the website: https://rna.informatik.uni-freiburg.de/IntaRNA/Input.jsp;jsessionid=2AE257E91E7276283F22CD77A5A8AA03
dynamic_efficiency	= true 
save_data           = false

flags = txtl_types.flags(positive_aminoacid, anti_shinedalgarno, dynamic_efficiency)

# Data import --------
mRNA 		= to_DataFrame(readmatrix("../data/input/mRNA_concentrations.xlsx", "mRNA_concentration", "A1:B2324"))
mRNA.mRNA 	= Float64.(mRNA.mRNA);

background_genes = String.(first(mRNA.gene, num_background_genes))
genes = vcat(heterologous_genes, background_genes)
println("Number of sequences: $(length(genes))") 

mRNA = subset(mRNA, :gene => ByRow(in(genes)))

## Import information about model sp ecies and sequences 
seqs, seq_names, hyb_file_dict, species_and_seqs, idx = species_and_sequences(genes, µ, sequence_file, hybrid_file, flags, seqs_to_use)

## Calculate efficiency factors 
efac_pos_aa_dict, efac_int_sd_dict, dynamic_efficiencies = efficiency_factor_initialization(seqs, seq_names, hyb_file_dict, species_and_seqs, flags)

## calculation of initial concentrations
y0, initial_concentrations = calculation_of_initial_concentration(species_and_seqs, seqs, seq_names, scale, idx);

## calculate and load kinetic parameters
p_met, p_seq, p_dyn = parameters(initial_concentrations, species_and_seqs, scale, µ);

## Release factor balance
RF_balance!(y0, initial_concentrations)

# Preintegration
## Definition of mass matrix
conserved = [idx.EFG, idx.EFD] # EFG, EFTu_GDP - Conserved species, Algebraic equations are given by mass balances

M = Matrix{Float64}(I, length(y0), length(y0)) # Creates unity matrix in the size of y0

M[conserved, :] .= 0; # setting mass matrix to zero to  remove time derivatives from those variables. This converts these differential equations into algebraic constraints


tags_pre = txtl_types.data_struct_preintegration(scale, idx, species_and_seqs, flags, dynamic_efficiencies, initial_concentrations, p_met, p_seq, p_dyn, mRNA.mRNA[1])

## Integration of the DAE system without DNA in the system 
y_steady_state = preintegration(tags_pre, y0, M);


# Elongation factor balance 
EF_balance!(tags_pre, y0, y_steady_state)


# Build states vector for multiple sequences 
y0_all = combined_states_vector(tags_pre, y_steady_state, seqs, seq_names)


# Prepare per-sequence tags (for use in ODE)
local_tags_list = generate_local_tags(seqs, seq_names, tags_pre, efac_pos_aa_dict, efac_int_sd_dict, mRNA);


# Integration
# PERFORMANCE FIX: Use Diagonal mass matrix instead of dense matrix
# For 100 sequences with ~22k states, dense matrix would be ~4GB and cause O(n³) scaling
n_all = length(y0_all)
mass_diag_all = ones(Float64, n_all)
mass_diag_all[conserved] .= 0.0
M = Diagonal(mass_diag_all)  # Much more memory-efficient: O(n) instead of O(n²)

tspan 	= (0.0, 100.0) ## Define time span for the simulation

p = generate_p(local_tags_list)

f 		= ODEFunction(ODE_parallel_distributed!, mass_matrix = M) ## Define ODEFunction to make use of mass matrix

prob 	= ODEProblem(f, y0_all, tspan, p); ## Create an ODE problem



# --- Timing the integration step ---
println("\n⏳ Integrating ODE system (threaded, $(length(seq_names)) sequences)...")



# PERFORMANCE FIX: Optimize save settings for large state vectors
# With ~22k states, saving every step is extremely expensive
# Only save start and end unless dense output is needed
sol = solve(prob, FBDF(autodiff=false), reltol = 1e-8, abstol = 1e-6
            save_everystep=false, save_start=false, save_end=true) ## Solve ODE system


println("Integration completed!")
save_results(sol, local_tags_list, save_data)


🔧 Julia started with 8 threads


Number of sequences: 2

🔍 PREINTEGRATION DIAGNOSTICS
Vector length: 835
Minimum value: 0.0
Maximum value: 129.5207905174607
Non-finite entries: 0
Values < -1e-9: 0
Values < 0: 0

✅ No negative species detected (threshold: -1e-9)


🔍 COMBINED STATES VECTOR DIAGNOSTICS
Vector length: 1308
Minimum value: 0.0
Maximum value: 43.241797675578404
Non-finite entries: 0
Values < -1e-9: 0
Values < 0: 0

✅ No negative species detected in y0_all (threshold: -1e-9)


⏳ Integrating ODE system (threaded, 2 sequences)...

🔍 SOLVER STATUS
Return code: Success
✅ Solver completed successfully

Integration completed!
20.06490264122067
20.084717450420307
0.011766836262505527
0.020946033381107912
2.3889801020818027
1.5445606500564322
0.6638077804586872
0.7464574530220764
0.21067229234558857
2.3626077232635274
0.09469122377607014
24.153954575798615
15.794536901035814
23.67830271624463
33.05784766401272
1.228969720864048e-6
0.4099853243461548
0.23650407388722466
3.3643967226304388
0.0004778210880152363
0.00030